In [1]:
import torch
import math
import random
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
## Example2 in pt1 done using pytorch

In [3]:
x1 = torch.tensor([2.0], dtype=torch.double, requires_grad=True)
x2 = torch.tensor([0.0], dtype=torch.double, requires_grad=True)
w1 = torch.tensor([-3.0], dtype=torch.double, requires_grad=True)
w2 = torch.tensor([1.0], dtype=torch.double, requires_grad=True)
b = torch.tensor([6.8813735870195432], dtype=torch.double, requires_grad=True)

n = x1*w1 + x2*w2 + b
o = torch.tanh(n)

print(o.data.item())
o.backward()


print('x2, ', x2.grad.item())
print('w2, ', w2.grad.item())
print('x1, ', x1.grad.item())
print('w1, ', w1.grad.item())

0.7071067811865476
x2,  0.49999999999999994
w2,  0.0
x1,  -1.4999999999999998
w1,  0.9999999999999999


In [4]:
# some more functions like substraction, division, negation, exponentiation are added to the Value class

In [5]:
class Value:
   def __init__(self, data, _children=(), _op='', label=''):
       self.data = data
       self.grad = 0.0
       self._backward = lambda:None
       self.label = label
       self._prev = set(_children)
       self._op = _op
       
   def __repr__(self):
      return f"(Value = {self.data})"
       
   def __add__(self, other):
      other = other if isinstance(other,Value) else Value(other)
      out = Value(self.data + other.data, (self, other), '+')
      def _backward():
          self.grad += 1.0 * out.grad
          other.grad += 1.0 * out.grad
      out._backward = _backward
      return out
       
   def __mul__(self, other):
      other = other if isinstance(other,Value) else Value(other)
      out = Value(self.data * other.data, (self, other), '*')
      def _backward():
          self.grad += other.data * out.grad
          other.grad += self.data * out.grad
      out._backward = _backward
      return out

   def __pow__(self, other):
    assert isinstance(other, (int, float)), "only supporting int/float powers for now"
    out = Value(self.data**other, (self,), f'**{other}')

    def _backward():
        self.grad += other * (self.data ** (other - 1)) * out.grad
    out._backward = _backward

    return out
  
   def __rmul__(self, other): # other * self
    return self * other

   def __truediv__(self, other): # self / other
    return self * other**-1

   def __neg__(self): # -self
    return self * -1

   def __sub__(self, other): # self - other
    return self + (-other)

   def __radd__(self, other): # other + self
    return self + other

       
   def tanh(self):
       x = self.data
       t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
       out = Value(t, (self,), 'tanh')
       def _backward():
          self.grad += (1 - t**2) * out.grad
       out._backward = _backward
       return out
       
   def backward(self):
       topo = []
       visited = set()
       def build_topo(v):
           if v not in visited:
               visited.add(v)
               for child in v._prev: 
                  build_topo(child)
               topo.append(v)
       build_topo(self)

       self.grad = 1.0
       for node in reversed(topo):
           node._backward()


In [6]:
# now we implement a Multi layer perceptron (MLP) Neural network from scratch
# Features: Scalar backpropagation via Value
# Note: The parameters() method gathers all trainable weights and biases for optimization.

In [7]:
class Neuron:
  
  def __init__(self, nin):
    self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
    self.b = Value(random.uniform(-1,1))
  
  def __call__(self, x):
    # w * x + b
    act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
    out = act.tanh()
    return out
  
  def parameters(self):
    return self.w + [self.b]

class Layer:
  
  def __init__(self, nin, nout):
    self.neurons = [Neuron(nin) for _ in range(nout)]
  
  def __call__(self, x):
    outs = [n(x) for n in self.neurons]
    return outs[0] if len(outs) == 1 else outs
  
  def parameters(self):
    return [p for neuron in self.neurons for p in neuron.parameters()]

class MLP:
  
  def __init__(self, nin, nouts):
    sz = [nin] + nouts
    self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
  
  def __call__(self, x):
    for layer in self.layers:
      x = layer(x)
    return x
  
  def parameters(self):
    return [p for layer in self.layers for p in layer.parameters()]


In [8]:
# Forward pass on a single 3D input vector

In [9]:
x = [2.0, 3.0, -1.0]
n = MLP(3, [4, 4, 1])  ## Network with 3 inputs, two 4-neuron hidden layers, and 1 output n(x)
n(x)

(Value = 0.9214596309986856)

In [10]:
#Defining a small training dataset (4 samples, 3 features each)

In [11]:
xs = [
  [2.0, 3.0, -1.0],
  [3.0, -1.0, 0.5],
  [0.5, 1.0, 1.0],
  [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0] # desired targets

In [12]:
for k in range(20):
  
  # forward pass
  ypred = [n(x) for x in xs]
  loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))
  
  # backward pass
  for p in n.parameters():
    p.grad = 0.0
  loss.backward()
  
  # update
  for p in n.parameters():
    p.data += -0.1 * p.grad
  
  print(k, loss.data)


0 5.441836051518724
1 0.18431678927790185
2 0.0808667814870251
3 0.0558866465891156
4 0.043485517497476596
5 0.03591347774592799
6 0.030754592957072467
7 0.02698840385919765
8 0.024104289998004878
9 0.021816713235108642
10 0.019952805969163826
11 0.01840145120179237
12 0.01708779056117383
13 0.01595944653412874
14 0.0149786092155429
15 0.014117260382684987
16 0.013354169614557801
17 0.012672937785046617
18 0.012060683943504801
19 0.011507140758198468


In [13]:
ypred

[(Value = 0.9572893871648721),
 (Value = -0.9540347838377904),
 (Value = -0.926292197859161),
 (Value = 0.9537690242783635)]